# Mander's Correlation Coefficients

In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "matplotlib",
#     "ndv[jupyter,vispy]",
#     "scikit-image",
#     "scipy",
#     "tifffile",
#     "imagecodecs",
# ]
# ///

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Description</mark>

In this notebook, we will explore how to implement **Mander's Correlation Coefficients** in Python, a common method for quantifying colocalization based on pixel intensities.

```{warning}
NOTE: This notebook aims to show how to practically implement these methods but does not aim to describe when to use this method. The images used have been selected to showcase the practical implementation of the methods.
```

```{note}
In this example, we will not perform any image processing steps before computing the Mander's Correlation Coefficients. However, when conducting a real colocalization analysis you should consider applying some image processing steps to clean the images before computing the Mander's Correlation Coefficients, such as background subtraction, flat-field correction, etc.
```

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Mander's Correlation Coefficients</mark>

Mander's correlation coefficients can be used to quantify the degree of colocalization between two channels (or images). These coefficients, M1 and M2, are calculated based on the pixel intensities of the two channels and for this reason, they are different from a simple area overlap.

**M1** measures the **fraction of channel 1 intensity that co-occurs with channel 2**:
- **Numerator**: Sum of channel 1 intensities in pixels where both channels are above their thresholds
- **Denominator**: Sum of all channel 1 intensities above threshold

**M2** measures the **fraction of channel 2 intensity that co-occurs with channel 1**:
- **Numerator**: Sum of channel 2 intensities in pixels where both channels are above their thresholds
- **Denominator**: Sum of all channel 2 intensities above threshold

<div> <img src="https://raw.githubusercontent.com/HMS-IAC/bobiac/main/_static/images/coloc/manders_slide.png" alt="Ilastik Logo" width="800"></div>

For this exercise, we will analyze an image of a HeLa cell stained with two fluorescent markers: **channel 1** labels **endosomes** and **channel 2** labels **lysosomes**. 

From a biological perspective, lysosomes are typically found within or closely associated with endosomal compartments, while endosomes have a broader cellular distribution. Based on this biology, we expect:

- **High M2 coefficient**: Most lysosomal signal should colocalize with endosomal regions
- **Lower M1 coefficient**: Only a subset of endosomal signal should colocalize with lysosomes, since endosomes are more widely distributed throughout the cell.

### <mark style="color: black; background-color: rgb(190,223,185); padding: 3px; border-radius: 5px;">Import Libraries</mark>

### <mark style="color: black; background-color: rgb(190,223,185); padding: 3px; border-radius: 5px;">Load and Visualize the Image</mark>

To compute Mander's Correlation Coefficients, we need **two separate images** (channels). 

What is the image shape? How do we split the channels?

### <mark style="color: black; background-color: rgb(190,223,185); padding: 3px; border-radius: 5px;">Calculate Numerators and Denominators for Mander's Correlation Coefficients</mark>

The first and key step is to calculate **$R_i^{}$** and **$G_i^{}$** and thus to select which areas of each channel we want to consider for the colocalization analysis. This means we first need to **threshold each images to select only the pixels we want to consider**.

It is therefore evident that Mander's Correlation Coefficients are **sensitive to thresholding**, the way you decide to threshold your images will have a large impact on the results.

For this example, we will first use a simple Otsu thresholding method and later in the notebook we will explore a more automated way of selecting the threshold.

We can plot the raw data and the masks in a 2x2 subplot to visualize the results of the thresholding.

Now that we have the mask for each channel, we can first **calculate the overlap mask** where both channels are above their respective thresholds, and then calculate **$R_i^{coloc}$** and **$G_i^{coloc}$**.

With the overlap mask, we can now calculate the **$R_i^{coloc}$** (*ch1_coloc*) and **$G_i^{coloc}$** (*ch2_coloc*) and the **numerator** for the Mander's Correlation Coefficients: **sum($R_i^{coloc}$)** and **sum($G_i^{coloc}$)**.

We can now **calculate the denominator** for the Manders coefficients.
<br>
The denominator is the sum of the pixel intensities in the overlap mask for each channel above their respective thresholds: **sum($R_i^{}$)** and **sum($G_i^{}$)**.

### <mark style="color: black; background-color: rgb(190,223,185); padding: 3px; border-radius: 5px;">Calculating Mander's Correlation Coefficients</mark>

Now with both numerators and denominators calculated, we can compute the Manders coefficients M1 and M2.

With Otsu thresholding for both channels, we obtain:

**M1=0.3496** and **M2=0.8975**

- **M1** indicates that approximately **35%** (0.3496) of channel 1's intensity colocalizes with channel 2. This means that about one-third of channel 1's signal overlaps with areas where channel 2 is also present above threshold.

- **M2** indicates that approximately **90%** (0.8975) of channel 2's intensity colocalizes with channel 1. This suggests that nearly all of channel 2's signal overlaps with areas where channel 1 is also present above threshold.

This asymmetry (M1 ≠ M2) is common and tells us that **channel 2 is largely contained within areas where channel 1 is present**, but **channel 1 extends beyond the regions where channel 2 is found**.

### <mark style="color: black; background-color: rgb(190,223,185); padding: 3px; border-radius: 5px;">Costes Auto-Threshold Method</mark>

As mentioned above, the Mender's Correlation Coefficients are sensitive to thresholding, so the way you decide to threshold your images will have a large impact on the results.

The function `costes_auto_threshold` below implements the [Costes auto-threshold method](https://pmc.ncbi.nlm.nih.gov/articles/PMC1304300/), which **automatically determines optimal threshold values for both channels**. 

The method works by finding threshold values where pixels *below* these thresholds show no statistical correlation (Pearson correlation coefficient ≈ 0). This approach helps objectively separate true signal from background noise.

The algorithm performs orthogonal linear regression between the two channels to establish their relationship, then iteratively tests threshold pairs derived from this regression to identify the optimal separation point between signal and background.

We can now try to compute the Mander's Correlation Coefficients using the Costes auto-threshold method.

In [ ]:
def costes_auto_threshold(
    ch1: np.ndarray,
    ch2: np.ndarray,
    num_thresholds: int = 100,
) -> tuple[float, float, float, float]:
    """
    Implementation of Costes auto-threshold method for colocalization analysis.

    Based on:
    Costes et al. "Automatic and quantitative measurement of protein-protein
    colocalization in live cells" Biophysical Journal 2004
    https://pmc.ncbi.nlm.nih.gov/articles/PMC1304300/

    The method finds thresholds where the Pearson correlation coefficient
    of pixels below the thresholds equals zero, indicating that pixels
    below these thresholds show no statistical correlation.

    Parameters
    -----------
    ch1: np.ndarray
        First channel image data (2D array).
    ch2: np.ndarray
        Second channel image data (2D array).
    num_thresholds: int
        Number of threshold values to test along the regression line. By default, 100.

    Returns
    ------
    tuple: (threshold_ch1, threshold_ch2, slope, intercept)
        Optimal thresholds for channel 1 and channel 2, slope and intercept of the
        regression line that relates the two channels.
    """

    def orthogonal_regression(x, y):
        """Perform orthogonal regression using PCA"""
        # Center the data
        x_centered = x - np.mean(x)
        y_centered = y - np.mean(y)

        # Stack data for PCA
        data = np.vstack([x_centered, y_centered]).T

        # Compute covariance matrix
        cov_matrix = np.cov(data.T)

        # Get eigenvalues and eigenvectors
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # The first principal component is the eigenvector with largest eigenvalue
        principal_component = eigenvectors[:, np.argmax(eigenvalues)]

        # Get slope from first principal component
        slope = principal_component[1] / principal_component[0]
        intercept = np.mean(y) - slope * np.mean(x)

        return slope, intercept

    # Flatten images for easier processing
    ch1_flat = ch1.flatten()
    ch2_flat = ch2.flatten()

    # If the min value is zero, consider only non-zero pixels
    if np.min(ch1_flat) == 0 or np.min(ch2_flat) == 0:
        mask = (ch1_flat > 0) & (ch2_flat > 0)
        ch1_masked = ch1_flat[mask]
        ch2_masked = ch2_flat[mask]
    else:
        ch1_masked = ch1_flat
        ch2_masked = ch2_flat

    if len(ch1_masked) == 0 or len(ch2_masked) == 0:
        return 0, 0, 0, 0

    # Perform orthogonal regression to find the relationship between channels
    # This gives us the line IG = a * IR + b using orthogonal regression as per Costes paper
    slope, intercept = orthogonal_regression(ch1_masked, ch2_masked)

    # Create threshold pairs along the regression line
    # Choose iteration strategy based on slope and dynamic range for better numerical stability
    max_ch1, max_ch2 = np.max(ch1_masked), np.max(ch2_masked)
    min_ch1, min_ch2 = np.min(ch1_masked), np.min(ch2_masked)

    range_ch1 = max_ch1 - min_ch1
    range_ch2 = max_ch2 - min_ch2

    best_thr_ch1 = best_thr_ch2 = 0
    best_correlation = 1.0

    # Choose which channel to iterate over based on:
    # 1. Slope magnitude (for numerical stability)
    # 2. Dynamic range (for better sampling)
    use_ch1_as_driver = True

    # Is the slope steep, iterate over ch2 for stability
    if abs(slope) > 1.5:
        use_ch1_as_driver = False
    # Elif slope is shallow, iterate over ch1 for stability
    elif abs(slope) < 0.67:
        use_ch1_as_driver = True
    # Otherwise use channel with larger range dynamic range
    else:
        use_ch1_as_driver = range_ch1 >= range_ch2

    if use_ch1_as_driver:
        # Iterate over ch1 thresholds, calculate ch2 from regression
        threshold_values = np.linspace(max_ch1, min_ch1, num_thresholds)

        for thr_ch1 in threshold_values:
            thr_ch2 = slope * thr_ch1 + intercept

            # Create mask for pixels below thresholds
            below_mask = (ch1_masked < thr_ch1) & (ch2_masked < thr_ch2)

            if np.sum(below_mask) < 10:  # Need minimum number of pixels
                continue

            # Calculate correlation for pixels below threshold
            ch1_below = ch1_masked[below_mask]
            ch2_below = ch2_masked[below_mask]

            if len(ch1_below) > 1 and np.std(ch1_below) > 0 and np.std(ch2_below) > 0:
                correlation, _ = pearsonr(ch1_below, ch2_below)

                # Find threshold where correlation is closest to zero
                if abs(correlation) < abs(best_correlation):
                    best_correlation = correlation
                    best_thr_ch1 = thr_ch1
                    best_thr_ch2 = thr_ch2

    else:
        # Iterate over ch2 thresholds, calculate ch1 from regression
        threshold_values = np.linspace(max_ch2, min_ch2, num_thresholds)

        for thr_ch2 in threshold_values:
            # Solve for ch1: ch1 = (ch2 - intercept) / slope
            if abs(slope) > 1e-10:  # Avoid division by zero
                thr_ch1 = (thr_ch2 - intercept) / slope
            else:
                continue

            # Create mask for pixels below thresholds
            below_mask = (ch1_masked < thr_ch1) & (ch2_masked < thr_ch2)

            if np.sum(below_mask) < 10:  # Need minimum number of pixels
                continue

            # Calculate correlation for pixels below threshold
            ch1_below = ch1_masked[below_mask]
            ch2_below = ch2_masked[below_mask]

            if len(ch1_below) > 1 and np.std(ch1_below) > 0 and np.std(ch2_below) > 0:
                correlation, _ = pearsonr(ch1_below, ch2_below)

                # Find threshold where correlation is closest to zero
                if abs(correlation) < abs(best_correlation):
                    best_correlation = correlation
                    best_thr_ch1 = thr_ch1
                    best_thr_ch2 = thr_ch2

    return best_thr_ch1, best_thr_ch2, slope, intercept

Now that we have the Costes thresholds, we can calculate the Mander's Correlation Coefficients using the Costes thresholds as we did for the Otsu thresholds.

How do thresholds and mask images compare to the Otsu thresholds?

### <mark style="color: black; background-color: rgb(190,223,185); padding: 3px; border-radius: 5px;">Summary</mark>

The Python implementation for calculating Mander's Correlation Coefficients is straightforward and concise, as demonstrated in the code below. However, it's crucial to remember that these coefficients are highly sensitive to thresholding methods. The choice of thresholding strategy will significantly impact your results, making careful threshold selection essential for accurate colocalization analysis.

Additionally, Mander's coefficients should not be interpreted as absolute values in isolation. Instead, it's always recommended to consider them in the context of comparisons between different conditions, controls, treatments, or experimental groups. The relative changes and ratios between conditions are often more meaningful than the absolute coefficient values themselves.


```python
# Create binary mask for channel 1 & 2 using a thresholding method of choice
threshold_ch1, threshold_ch2 = threshold_method(ch1, ch2)
image_1_mask = ch1 > threshold_ch1
image_2_mask = ch2 > threshold_ch2

# Find pixels that are above threshold in both channels
overlap_mask = image_1_mask & image_2_mask

# Extract channel 1 & 2 intensities only from overlapping regions
ch1_coloc = ch1[overlap_mask]
ch2_coloc = ch2[overlap_mask]

# Extract all channel 1 & 2 intensities above threshold
ch1_tr = ch1[image_1_mask]
ch2_tr = ch2[image_2_mask]

# Calculate total intensity of channel 1 & 2 above threshold
sum_ch1_tr = np.sum(ch1_tr)
sum_ch2_tr = np.sum(ch2_tr)

# M1: fraction of channel 1 intensity that colocalizes with channel 2
M1 = np.sum(ch1_coloc) / sum_ch1_tr
# M2: fraction of channel 2 intensity that colocalizes with channel 1
M2 = np.sum(ch2_coloc) / sum_ch2_tr
```
